In [180]:
import sys
from mlflow.tracking import MlflowClient

import numpy as np
import pandas as pd


from tqdm import tqdm

sys.path.append("../.")


pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 400)
pd.set_option("display.float_format", lambda x: "%.4f" % x)


%reload_ext autoreload
%autoreload 2

In [181]:
# Initialize MLflow Client
client = MlflowClient(tracking_uri="http://potato.felk.cvut.cz:2222")

experiment_names = [
    "pelesjak_getml_xgboost",
    "pelesjak_lightgbm_aggregate_hyperparams",
    "pelesjak_lightgbm_hyperparams",
    "pelesjak_linear_dbformer_hyperparams",
    "pelesjak_linear_sage_hyperparams",
    "pelesjak_linear_tabular_aggregate_hyperparams",
    "pelesjak_linear_tabular_hyperparams",
    "pelesjak_resnet_sage_hyperparams",
    "pelesjak_resnet_tabular_aggregate_hyperparams",
    "pelesjak_resnet_tabular_hyperparams",
]

for experiment_name in experiment_names:
    experiment = client.get_experiment_by_name(experiment_name)
    if experiment is None:
        raise ValueError(f"Experiment '{experiment_name}' not found.")
    experiment_id = experiment.experiment_id

    # Get all runs for the experiment
    runs = client.search_runs(experiment_ids=[experiment_id], max_results=10000)

    runs_dict = []
    for r in tqdm(runs.to_list()):
        r_info = r.info.__dict__
        r_data = {k: v for d in r.data.to_dictionary().values() for k, v in d.items()}

        runs_dict.append({**r_info, **r_data})

    df = pd.DataFrame(runs_dict)

    df.to_csv(f"./data/{experiment_name}.csv")

100%|██████████| 400/400 [00:00<00:00, 282159.70it/s]


In [197]:
dataset_tasks_index = [
    ("ctu-ergastf1", "ergastf1-original"),
    ("ctu-expenditures", "expenditures-original"),
    ("ctu-geneea", "geneea-original"),
    ("ctu-geneea", "geneea-temporal"),
    ("ctu-hepatitis", "hepatitis-original"),
    ("ctu-imdb", "imdb-original"),
    ("ctu-mondial", "mondial-original"),
    ("ctu-movielens", "movielens-original"),
    ("ctu-musklarge", "musklarge-original"),
    ("ctu-musksmall", "musksmall-original"),
    ("ctu-mutagenesis", "mutagenesis-original"),
    ("ctu-ncaa", "ncaa-original"),
    ("ctu-studentloan", "studentloan-original"),
    ("rel-amazon", "item-churn"),
    ("rel-amazon", "user-churn"),
    ("rel-avito", "user-clicks"),
    ("rel-avito", "user-visits"),
    ("rel-f1", "driver-dnf"),
    ("rel-f1", "driver-top3"),
    ("rel-stack", "user-badge"),
    ("rel-stack", "user-engagement"),
    ("rel-trial", "study-outcome"),
    ("ctu-accidents", "accidents-original"),
    ("ctu-accidents", "accidents-temporal"),
    ("ctu-craftbeer", "craftbeer-original"),
    ("ctu-dallas", "dallas-original"),
    ("ctu-dallas", "dallas-temporal"),
    ("ctu-diabetes", "diabetes-original"),
    ("ctu-financial", "financial-original"),
    ("ctu-financial", "financial-temporal"),
    ("ctu-genes", "genes-original"),
    ("ctu-hockey", "hockey-original"),
    ("ctu-legalacts", "legalacts-original"),
    ("ctu-legalacts", "legalacts-temporal"),
    ("ctu-premiereleague", "premiereleague-original"),
    ("ctu-premiereleague", "premiereleague-temporal"),
    ("ctu-tpcd", "tpcd-original"),
    ("ctu-webkp", "webkp-original"),
]

### Get overall results


In [198]:
def get_topn_results(df, n=5, num_layers=None):
    if num_layers is not None and "num_layers" in df.columns:
        df = df[df["num_layers"] == num_layers].drop(columns=["num_layers"])
    df = df[
        [
            "dataset_name",
            "task_name",
            "val_roc_auc",
            "test_roc_auc",
            "val_macro_f1",
            "test_macro_f1",
        ]
    ]

    df = df[
        (
            (df.val_roc_auc >= 0)
            & (df.val_roc_auc <= 1)
            & (df.test_roc_auc >= 0)
            & (df.test_roc_auc <= 1)
        )
        | (
            (df.val_macro_f1 >= 0)
            & (df.val_macro_f1 <= 1)
            & (df.test_macro_f1 >= 0)
            & (df.test_macro_f1 <= 1)
        )
    ]
    df_topn = []
    for val_metric in ["val_macro_f1", "val_roc_auc"]:
        # Drop rows where the validation metric is missing
        df_filtered = df.dropna(subset=[val_metric])

        # For each (dataset_name, task_name), get top N rows by validation metric
        df_topn.append(
            df_filtered.sort_values(val_metric, ascending=False)
            .groupby(["dataset_name", "task_name"], group_keys=False)
            .head(n)
        )

    return (
        pd.concat(df_topn)
        .groupby(["dataset_name", "task_name"])
        .aggregate(["mean", "std"])
        .reindex(index=dataset_tasks_index)
    )

In [205]:
lightgbm_df = pd.read_csv("./data/pelesjak_lightgbm_hyperparams.csv")
lightgbm_agg_df = pd.read_csv("./data/pelesjak_lightgbm_aggregate_hyperparams.csv")

prop_df = pd.read_csv("./data/pelesjak_getml_xgboost.csv")
prop_df["_start_time"] = pd.to_datetime(prop_df["_start_time"], unit="ms")
prop_df_new = prop_df[(prop_df["_start_time"] > pd.Timestamp("2025-10-08 13:30:00"))]
prop_df_old = prop_df[prop_df["task_name"].str.contains("original")]
prop_df = pd.concat([prop_df_new, prop_df_old])


tabular_df = pd.read_csv("./data/pelesjak_resnet_tabular_hyperparams.csv")
tabular_agg_df = pd.read_csv("./data/pelesjak_resnet_tabular_aggregate_hyperparams.csv")

linear_sage_raw_df = pd.read_csv("./data/pelesjak_linear_sage_hyperparams.csv")
resnet_sage_raw_df = pd.read_csv("./data/pelesjak_resnet_sage_hyperparams.csv")
dbformer_raw_df = pd.read_csv("./data/pelesjak_linear_dbformer_hyperparams.csv")

N = 5
lightgbm_df = get_topn_results(lightgbm_df, n=N)
lightgbm_agg_df = get_topn_results(lightgbm_agg_df, n=N)
prop_df = get_topn_results(prop_df, n=N)
tabular_df = get_topn_results(tabular_df, n=N)
tabular_agg_df = get_topn_results(tabular_agg_df, n=N)
linear_sage_df = get_topn_results(linear_sage_raw_df, n=N)
linear_sage_l1_df = get_topn_results(linear_sage_raw_df, n=N, num_layers=1)
resnet_sage_df = get_topn_results(resnet_sage_raw_df, n=N)
resnet_sage_l1_df = get_topn_results(resnet_sage_raw_df, n=N, num_layers=1)
dbformer_df = get_topn_results(dbformer_raw_df, n=N)
dbformer_l1_df = get_topn_results(dbformer_raw_df, n=N, num_layers=1)

In [206]:
overall_df_mean = pd.concat(
    [
        lightgbm_df,
        lightgbm_agg_df,
        prop_df,
        tabular_df,
        tabular_agg_df,
        linear_sage_df,
        linear_sage_l1_df,
        resnet_sage_df,
        resnet_sage_l1_df,
        dbformer_df,
        dbformer_l1_df,
    ],
    axis=1,
    keys=(
        "LightGBM",
        "LightGBM Agg",
        "GetML XGBoost",
        "Tabular ResNet",
        "Tabular ResNet Agg",
        "Linear SAGE",
        "Linear SAGE L1",
        "ResNet SAGE",
        "ResNet SAGE L1",
        "DBFormer",
        "DBFormer L1",
    ),
)
overall_df_mean = overall_df_mean.reset_index().rename(
    {"dataset_name": "Dataset", "task_name": "Task"}, axis=1
)

overall_df_mean.set_index(["Dataset", "Task"], drop=True, inplace=True)

bin_overall = (
    overall_df_mean.swaplevel(0, 1, axis=1)[["val_roc_auc", "test_roc_auc"]]
    .dropna(how="all")
    .swaplevel(0, 2, axis=1)["mean"]
    .sort_index(axis=1, level=[0, 1], ascending=[True, False])
    .swaplevel(0, 1, axis=1)
    .rename(
        columns={
            "val_roc_auc": "val",
            "test_roc_auc": "test",
        }
    )
    .swaplevel(0, 1, axis=1)
)[
    [
        "LightGBM",
        "LightGBM Agg",
        "GetML XGBoost",
        "Tabular ResNet",
        "Tabular ResNet Agg",
        "Linear SAGE",
        "Linear SAGE L1",
        "ResNet SAGE",
        "ResNet SAGE L1",
        "DBFormer",
        "DBFormer L1",
    ]
]

mlt_overall = (
    overall_df_mean.swaplevel(0, 1, axis=1)[["val_macro_f1", "test_macro_f1"]]
    .dropna(how="all")
    .swaplevel(0, 2, axis=1)["mean"]
    .sort_index(axis=1, level=[0, 1], ascending=[True, False])
    .swaplevel(0, 1, axis=1)
    .rename(
        columns={
            "val_macro_f1": "val",
            "test_macro_f1": "test",
        }
    )
    .swaplevel(0, 1, axis=1)
)

overall_df_mean = pd.concat([bin_overall, mlt_overall], axis=0)

overall_df_mean = overall_df_mean.round(3)

overall_df_mean_all = overall_df_mean
overall_df_mean = overall_df_mean[
    [
        "LightGBM",
        "GetML XGBoost",
        "Tabular ResNet",
        "Linear SAGE",
        "ResNet SAGE",
        "DBFormer",
    ]
]

ranks_total = (
    overall_df_mean.swaplevel(0, 1, axis=1)["test"]
    .dropna(how="all")
    .rank(ascending=False, axis=1, na_option="bottom", method="average")
)


ranks_merged = overall_df_mean.swaplevel(0, 1, axis=1)["test"].dropna(how="all")
ranks_merged["RDL"] = ranks_merged[["Linear SAGE", "ResNet SAGE", "DBFormer"]].max(axis=1)
ranks_merged = ranks_merged[["LightGBM", "GetML XGBoost", "Tabular ResNet", "RDL"]].rank(
    ascending=False, axis=1, na_option="bottom", method="average"
)
ranks_merged_vals = np.zeros(6)
ranks_merged_vals[:4] = ranks_merged.mean(axis=0).values
overall_df_mean = overall_df_mean.swaplevel(0, 1, axis=1)
overall_df_mean.loc[("Avg. Rank", ""), "test"] = ranks_total.mean(axis=0).values
overall_df_mean.loc[("Avg. Rank Merged", ""), "test"] = ranks_merged_vals
overall_df_mean = overall_df_mean.swaplevel(0, 1, axis=1)


def highlight_max_val_test(row):
    is_val = row.index.get_level_values(1) == "val"
    is_test = row.index.get_level_values(1) == "test"

    styles = pd.Series("", index=row.index)

    val_max = row[is_val].max()
    styles.loc[(row == val_max) & is_val] = "font-weight: bold;"

    test_max = row[is_test].max()
    styles.loc[(row == test_max) & is_test] = "font-weight: bold;"

    return styles


overall_df_mean = overall_df_mean.reset_index()
ctu_mask = overall_df_mean["Dataset"].str.contains("ctu")
overall_df_mean.loc[ctu_mask, "Task"] = (
    overall_df_mean.loc[ctu_mask, "Task"].str.split("-").str[1]
).values
overall_df_mean["Dataset"] = (
    overall_df_mean["Dataset"].str.replace("ctu-", "").str.replace("rel-", "")
)

overall_df_mean.set_index(["Dataset", "Task"], drop=True, inplace=True)
overall_df_mean.style.apply(highlight_max_val_test, axis=1).format(precision=3).to_latex(
    "./data/tables/class-overall.tex", convert_css=True, column_format="llcccccccccccc"
)
overall_df_mean.style.apply(highlight_max_val_test, axis=1).format(precision=3)

In [207]:
compare_tasks_index = [
    ("ctu-ergastf1", "ergastf1-original"),
    ("ctu-hepatitis", "hepatitis-original"),
    ("ctu-mondial", "mondial-original"),
    ("ctu-movielens", "movielens-original"),
    ("ctu-ncaa", "ncaa-original"),
    ("ctu-studentloan", "studentloan-original"),
    ("rel-amazon", "user-churn"),
    ("rel-avito", "user-clicks"),
    ("rel-avito", "user-visits"),
    ("rel-stack", "user-badge"),
    ("ctu-accidents", "accidents-original"),
    ("ctu-accidents", "accidents-temporal"),
    ("ctu-craftbeer", "craftbeer-original"),
    ("ctu-diabetes", "diabetes-original"),
    ("ctu-genes", "genes-original"),
    ("ctu-premiereleague", "premiereleague-original"),
    ("ctu-premiereleague", "premiereleague-temporal"),
    ("ctu-tpcd", "tpcd-original"),
    ("ctu-webkp", "webkp-original"),
]
tabular_compare_df_mean = overall_df_mean_all[
    [
        "LightGBM",
        "LightGBM Agg",
        "Tabular ResNet",
        "Tabular ResNet Agg",
        "Linear SAGE",
        "Linear SAGE L1",
        "ResNet SAGE",
        "ResNet SAGE L1",
        "DBFormer",
        "DBFormer L1",
    ]
]
rdl_max = tabular_compare_df_mean[
    [("Linear SAGE", "val"), ("ResNet SAGE", "val"), ("DBFormer", "val")]
].idxmax(axis=1)
rdl_max_l1 = tabular_compare_df_mean[
    [("Linear SAGE L1", "val"), ("ResNet SAGE L1", "val"), ("DBFormer L1", "val")]
].idxmax(axis=1)
rdl_max_vals = []
for row in tabular_compare_df_mean.iterrows():
    rdl_max_vals.append(row[1][rdl_max[row[0]][0]])
rdl_max_vals_l1 = []
for row in tabular_compare_df_mean.iterrows():
    rdl_max_vals_l1.append(row[1][rdl_max_l1[row[0]][0]])

rdl_max = pd.concat(rdl_max_vals, axis=1).T
rdl_max_l1 = pd.concat(rdl_max_vals_l1, axis=1).T
tabular_compare_df_mean[("RDL", "val")] = rdl_max["val"]
tabular_compare_df_mean[("RDL L1", "val")] = rdl_max_l1["val"]
tabular_compare_df_mean[("RDL", "test")] = rdl_max["test"]
tabular_compare_df_mean[("RDL L1", "test")] = rdl_max_l1["test"]
tabular_compare_df_mean = tabular_compare_df_mean.loc[
    compare_tasks_index,
    ["LightGBM", "LightGBM Agg", "Tabular ResNet", "Tabular ResNet Agg", "RDL L1", "RDL"],
]


def highlight_improvement(row):
    is_val = row.index.get_level_values(1) == "val"
    val_max = row[is_val].max()

    is_test = row.index.get_level_values(1) == "test"
    test_max = row[is_test].max()

    styles = pd.Series("", index=row.index)

    if row.loc[("LightGBM", "val")] + 0.05 < row.loc[("LightGBM Agg", "val")]:
        styles.loc[("LightGBM Agg", "val")] = "font-weight: bold;"
        if row.loc[("LightGBM Agg", "val")] == val_max:
            styles.loc[("LightGBM Agg", "val")] += "text-decoration: underline;"

    if row.loc[("LightGBM", "test")] + 0.05 < row.loc[("LightGBM Agg", "test")]:
        styles.loc[("LightGBM Agg", "test")] = "font-weight: bold;"
        if row.loc[("LightGBM Agg", "test")] == test_max:
            styles.loc[("LightGBM Agg", "test")] += "text-decoration: underline;"

    if row.loc[("Tabular ResNet", "val")] + 0.05 < row.loc[("Tabular ResNet Agg", "val")]:
        styles.loc[("Tabular ResNet Agg", "val")] = "font-weight: bold;"
        if row.loc[("Tabular ResNet Agg", "val")] == val_max:
            styles.loc[("Tabular ResNet Agg", "val")] += "text-decoration: underline;"

    if row.loc[("Tabular ResNet", "test")] + 0.05 < row.loc[("Tabular ResNet Agg", "test")]:
        styles.loc[("Tabular ResNet Agg", "test")] = "font-weight: bold;"
        if row.loc[("Tabular ResNet Agg", "test")] == test_max:
            styles.loc[("Tabular ResNet Agg", "test")] += "text-decoration: underline;"

    return styles


tabular_compare_df_mean = tabular_compare_df_mean.reset_index()
ctu_mask = tabular_compare_df_mean["Dataset"].str.contains("ctu")
tabular_compare_df_mean.loc[ctu_mask, "Task"] = (
    tabular_compare_df_mean.loc[ctu_mask, "Task"].str.split("-").str[1]
).values
tabular_compare_df_mean["Dataset"] = (
    tabular_compare_df_mean["Dataset"].str.replace("ctu-", "").str.replace("rel-", "")
)

tabular_compare_df_mean.set_index(["Dataset", "Task"], drop=True, inplace=True)

tabular_compare_df_mean.style.apply(highlight_improvement, axis=1).format(
    precision=3
).to_latex(
    "./data/tables/class-tabular.tex",
    convert_css=True,
    column_format="lccccccccccccc",
)
tabular_compare_df_mean.style.apply(highlight_improvement, axis=1).format(precision=3)

/tmp/ipykernel_22422/3048584857.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tabular_compare_df_mean[("RDL", "val")] = rdl_max["val"]


In [208]:
dataset_info = pd.read_csv("../redelex/datasets/dataset-info.csv")
task_info = pd.read_csv("../redelex/tasks/task-info.csv")

_joined_info = task_info.merge(dataset_info, how="left", on=["dataset"]).set_index(
    ["dataset", "task"], drop=True
)
joined_info = _joined_info.select_dtypes(include="number")
joined_info = joined_info.loc[dataset_tasks_index]

ordered_features = {
    "n_tables": "#Tab.",
    "n_fks": "#FK",
    "n_factual_cols": "#Factual",
    "cat_col": "#Cat.",
    "num_col": "#Num.",
    "text_col": "#Text",
    "time_col": "#Time",
    "total_n_tuples": "#Rows",
    "total_n_fk_edges": "#Links",
    "schema_diameter": "Diameter",
    "one_to_one": "1-to-1",
    "one_to_many": "1-to-M",
    "n_train": "#Train",
    "entity_fact_cols": "#T. Factual",
    "target_categorical": "T. Cat.",
    "target_numerical": "T. Num.",
    "target_text_embedded": "T. Text",
    "target_timestamp": "T. Time",
    "entity_eccentricity": "Eccentricity",
    "sample_density": "Density",
}
joined_info = joined_info[ordered_features.keys()].rename(columns=ordered_features)
all_joined_info = _joined_info[ordered_features.keys()].rename(columns=ordered_features)

In [211]:
best_ranks = ranks_total.min(axis=1)
rdl_best = ranks_total[
    (ranks_total["Linear SAGE"] == best_ranks)
    | (ranks_total["ResNet SAGE"] == best_ranks)
    | (ranks_total["DBFormer"] == best_ranks)
]

tabular_best = ranks_total[
    (ranks_total["Tabular ResNet"] == best_ranks) | (ranks_total["LightGBM"] == best_ranks)
]

prop_best = ranks_total[(ranks_total["GetML XGBoost"] == best_ranks)]

rdl_info = joined_info.loc[rdl_best.index]
tabular_info = joined_info.loc[tabular_best.index]
prop_info = joined_info.loc[prop_best.index]

col_map = {0.25: "Q1", 0.5: "Median", 0.75: "Q3"}
base_info = all_joined_info.quantile([0.5]).T.rename(columns=col_map)
prop_info = prop_info.quantile([0.25, 0.5, 0.75]).T.rename(columns=col_map)
rdl_info = rdl_info.quantile([0.25, 0.5, 0.75]).T.rename(columns=col_map)
tabular_info = tabular_info.quantile([0.25, 0.5, 0.75]).T.rename(columns=col_map)

features_summary = pd.concat(
    [base_info, rdl_info, prop_info, tabular_info],
    axis=1,
    keys=["Base", "RDL", "Propositional.", "Tabular Learning"],
)

features_summary.to_latex(
    "./data/tables/database-characteristics.tex",
    column_format="lcccccccccc",
    float_format="%.3f",
)
features_summary.style.format(precision=3)